# 資料契約唯讀稽核 — 2026-09-20
範圍：本機 backfill-small-v2.sqlite3，不代表正式 Firestore。
SQL/檢查邏輯位於 repository 內的 audit_database_contract.py，使用 mode=ro。
結果：182 facts；6 筆月期間錯配、12 筆來源錯配（兩群重疊），上線閘門 BLOCKED。
沒有修改原始資料。缺少本機樣本檔時應明確失敗，不建立空資料庫。


In [1]:
from pathlib import Path
import importlib.util
import json

# Locate repository whether run from its root or this notebook directory.
root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "fintrust_backend/scripts/audit_database_contract.py").is_file())
spec = importlib.util.spec_from_file_location("contract_audit", root / "fintrust_backend/scripts/audit_database_contract.py")
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
result = module.audit(root / "fintrust_backend/data/backfill-small-v2.sqlite3")
summary = {
    "database": "fintrust_backend/data/backfill-small-v2.sqlite3",
    "rows": {name: item["rows"] for name, item in result["profile"].items()},
    "checks": result["checks"], "classification": result["classification"],
    "findings": result["findings"], "gate": result["gate"],
}
print(json.dumps(summary, ensure_ascii=False, indent=2))


{
  "database": "fintrust_backend/data/backfill-small-v2.sqlite3",
  "rows": {
    "analysis_runs": 4,
    "calculated_metrics": 448,
    "companies": 96,
    "financial_filings": 10,
    "ingestion_runs": 4,
    "latest_analysis_snapshots": 2,
    "normalized_financial_facts": 182,
    "official_events": 0,
    "rule_results": 80
  },
  "checks": {
    "sqlite_integrity": "ok",
    "fact_count": 182,
    "monthly_fact_count": 6,
    "blank_source_count": 0,
    "demo_fact_count": 0,
    "duplicate_fact_keys": 0,
    "orphan_company_facts": 0,
    "orphan_metric_runs": 0,
    "orphan_snapshot_runs": 0
  },
  "classification": [
    {
      "subindustry_confidence": "medium",
      "count": 92
    },
    {
      "subindustry_confidence": "reviewed",
      "count": 4
    }
  ],
  "findings": [
    {
      "code": "monthly_period_mismatch",
      "severity": "high",
      "count": 6,
      "fact_ids": [
        "a807fb05ae71e663980ac27f00a2c406250671f1",
        "6bacfafa83e18e42d880d80b0